In [9]:
import pandas as pd
import warnings
import numpy as np
warnings.filterwarnings("ignore")

In [10]:
from pymongo import MongoClient
from dotenv import load_dotenv
import os

# Load dataset

In [17]:
load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")

client = MongoClient(MONGO_URI)
db=client["PolymerPrediction"]
collection=db["biceranoTg"]
target="Tg"
#print(collection.count_documents({}))

In [18]:
docs = list(collection.find({}))
df = pd.DataFrame(docs)

# espando il dizionario "properties"
prop_df = pd.json_normalize(df["properties"])

# integro nel dataframe
df = pd.concat([df.drop(columns=["properties"]), prop_df], axis=1)

df = df.set_index("_id")
df.index = df.index.astype(str)

# rimuovo righe senza SMILES, Tg o con SMILES malformato
df = df.dropna(subset=["smiles", target])
df = df[~df["smiles"].str.contains(r"\[\[", na=False)].copy()
df.describe()

,Tg
count,304.000000
mean,360.565789
std,114.734917
min,130.000000
25%,272.000000
50%,359.000000
75%,418.250000
max,685.000000


In [34]:
y = df[target]
X = pd.read_csv("chemberta_embeddings.csv", sep=",").set_index("_id")

print("Shape X:", X.shape)
print("Shape y:", y.shape)
X.sample(10)


Shape X: (304, 768)
Shape y: (304,)


,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,emb_9,...,emb_758,emb_759,emb_760,emb_761,emb_762,emb_763,emb_764,emb_765,emb_766,emb_767
_id,,,,,,,,,,,,,,,,,,,,,
698ddd4a8603d1724287a2d3,0.870667,0.343762,0.279760,-0.481438,-0.421132,-0.707375,-0.343450,-0.373640,0.185759,0.263078,...,-0.052830,-0.294843,-0.513948,-1.215424,-0.437064,-0.057193,-0.719834,-0.468621,-0.365066,1.183160
698ddd4a8603d1724287a32e,0.419180,0.914982,0.432835,-0.914055,-0.026602,-0.245183,-0.446754,-0.557457,-0.739408,0.138673,...,-0.512444,0.424263,0.330587,-0.661958,-0.160573,-0.803096,-0.273890,-0.546282,0.033919,0.353973
698ddd4a8603d1724287a3b1,0.257247,-0.224399,0.036322,-0.337142,0.216936,0.395993,-0.524928,-0.011543,-0.249351,-0.328086,...,0.495926,0.909677,0.085976,-1.288436,-0.753313,0.181734,0.028241,0.553257,-0.250529,0.135365
698ddd4a8603d1724287a3c0,0.265980,-0.125669,-0.285499,-0.239607,0.336271,0.121546,-0.225165,-0.056774,-0.561070,-0.532219,...,0.669661,0.989066,0.628482,-1.143150,-0.765309,0.047290,-0.047866,0.525021,0.022923,-0.073565
698ddd4a8603d1724287a2c8,0.569440,0.436834,0.349882,0.129735,-0.286562,0.047977,-0.533190,-0.120368,-0.519440,-0.285706,...,0.160926,0.258876,-0.229365,-0.773453,-0.612372,-0.560739,-0.403449,-0.874879,0.049014,0.895539
698ddd4a8603d1724287a37c,0.545483,0.093512,0.193594,-0.342544,0.360307,0.760928,-0.065611,-0.518655,-0.677032,-0.219317,...,-0.002481,0.849399,-0.668662,-0.995615,-0.695316,0.618045,-0.263544,0.285780,-0.542837,0.480022
698ddd4a8603d1724287a317,0.381339,0.556828,-0.002554,-0.296402,0.087944,-0.149526,-0.192368,-0.031651,-0.587641,0.038620,...,0.046616,0.412193,0.053547,-0.855640,-0.622873,-0.639141,-0.668434,-0.240950,-0.007584,0.554907
698ddd4a8603d1724287a2e4,0.614165,0.555941,0.207525,-0.831004,-0.193386,-0.998305,-0.573438,0.082283,-0.332644,0.128939,...,-0.527387,0.158069,0.392791,-1.332536,-0.213555,-1.169554,-1.217331,-0.528993,-0.034531,0.842743
698ddd4a8603d1724287a2ee,0.393165,0.814986,0.264739,-0.951905,-0.017495,-0.230723,-0.469396,-0.615164,-0.684348,0.125369,...,-0.598779,0.422224,0.287256,-0.592247,-0.269301,-0.723178,-0.246857,-0.487535,-0.003627,0.344747


# Gradient Boosting baseline

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

random_state=25

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=random_state)

model_base = GradientBoostingRegressor(random_state=random_state)
model_base.fit(X_train, y_train)

pred_base = model_base.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred_base))
print("R2:", r2_score(y_test, pred_base))


MAE: 35.034965610068014
R2: 0.7591501605429914


# CV

In [23]:
from sklearn.model_selection import GridSearchCV, KFold

param_grid = {
    "n_estimators": [100, 200, 400],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "subsample": [0.8, 1.0]
}

gb = GradientBoostingRegressor(random_state=random_state)

# Cross-validation 5 fold
cv = KFold(n_splits=5, shuffle=True, random_state=random_state)

# Grid search
grid_search = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

# Risultati tuning
print("Best parameters found:")
print(grid_search.best_params_)

print("\nBest CV MAE:")
print(-grid_search.best_score_)

# Valutazione finale su test set
best_model = grid_search.best_estimator_
pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print("\nPerformance on Test Set:")
print(f"MAE: {mae}")
print(f"R² : {r2}")

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best parameters found:
{'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 400, 'subsample': 0.8}

Best CV MAE:
37.67368427168931

Performance on Test Set:
MAE: 33.66667938456785
R² : 0.7983253422547822


# Reduced embeddings

In [24]:
X_red = pd.read_csv("chemberta_embeddings_reduced.csv", sep=",").set_index("_id")

print("Shape X:", X_red.shape)
print("Shape y:", y.shape)
X_red.sample(10)


Shape X: (304, 50)
Shape y: (304,)


,pc_0,pc_1,pc_2,pc_3,pc_4,pc_5,pc_6,pc_7,pc_8,pc_9,...,pc_40,pc_41,pc_42,pc_43,pc_44,pc_45,pc_46,pc_47,pc_48,pc_49
_id,,,,,,,,,,,,,,,,,,,,,
698ddd4a8603d1724287a2e5,2.571699,-4.564453,-0.561356,0.785991,1.773352,0.127363,0.780920,0.377005,0.189540,-0.419672,...,0.489488,-0.103510,-0.151144,-0.638339,-0.129346,-0.010666,0.331819,0.362795,0.103108,-0.414995
698ddd4a8603d1724287a32b,8.418379,0.642715,-2.409473,-1.220300,-0.355368,-2.055792,2.473769,-2.295770,0.120357,-0.154120,...,0.224399,-0.316178,-0.106468,0.233718,0.643375,-0.095782,0.012040,-0.221606,0.035451,-0.239770
698ddd4a8603d1724287a303,0.886890,6.263260,-6.138596,0.823851,0.858072,3.111480,-1.576668,2.054966,-1.059602,-0.004110,...,-0.150268,-0.013771,-0.048348,-0.090986,0.272744,-0.131821,-0.127883,-0.405010,0.019838,0.326008
698ddd4a8603d1724287a2e0,5.963215,-0.820692,3.059162,0.551861,-1.854574,0.452572,-1.583690,-1.873274,0.095978,1.941587,...,0.408490,0.236329,-0.515701,0.102800,0.178587,0.236825,0.275566,0.061755,0.365326,0.090258
698ddd4a8603d1724287a30d,-0.539643,5.128990,4.437948,1.945075,0.852567,-1.413647,0.638843,0.950173,-1.252549,1.059894,...,-0.132712,0.035508,0.051377,-0.190127,-0.167112,0.002914,0.071422,-0.233562,-0.161066,0.181971
698ddd4a8603d1724287a308,5.376227,-0.831090,3.454350,1.817708,-1.810738,1.396742,-0.730462,-1.845955,0.243914,1.597282,...,0.315131,0.083415,-0.409548,0.323602,0.127756,0.344838,0.458597,0.180461,0.093757,-0.044067
698ddd4a8603d1724287a340,-1.925756,-1.457238,2.062190,4.886402,-0.284358,1.197891,0.734477,-2.733848,1.789906,-0.100670,...,0.373598,-0.005034,0.118552,0.335330,0.261921,-0.094175,0.082630,0.153738,-0.002412,0.311823
698ddd4a8603d1724287a29c,10.431001,-0.164577,-4.317696,-2.134494,-1.429927,-3.876562,2.362652,-2.254263,-0.063910,0.791862,...,-0.087417,0.468386,0.279918,-0.100096,0.112664,0.202250,-0.046105,0.038819,0.244750,0.136771
698ddd4a8603d1724287a3b2,-6.071821,-3.652609,-2.851501,1.045238,-0.987697,1.476924,2.418134,-2.343423,-2.601526,0.134668,...,-0.378551,-0.062139,0.105804,-0.370824,-0.089729,0.349565,0.250985,-0.070680,0.288157,0.239580


In [25]:
from sklearn.model_selection import GridSearchCV, KFold

param_grid = {
    "n_estimators": [100, 200, 400],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "subsample": [0.8, 1.0]
}

gb = GradientBoostingRegressor(random_state=random_state)
X_train_red, X_test_red, y_train, y_test = train_test_split(X_red, y, test_size=0.2,random_state=random_state)

# Cross-validation 5 fold
cv = KFold(n_splits=5, shuffle=True, random_state=random_state)

# Grid search
grid_search = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_red, y_train)

# Risultati tuning
print("Best parameters found:")
print(grid_search.best_params_)

print("\nBest CV MAE:")
print(-grid_search.best_score_)

# Valutazione finale su test set
best_model = grid_search.best_estimator_
pred_red = best_model.predict(X_test_red)

mae = mean_absolute_error(y_test, pred_red)
r2 = r2_score(y_test, pred_red)

print("\nPerformance on Test Set:")
print(f"MAE: {mae}")
print(f"R² : {r2}")

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best parameters found:
{'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 400, 'subsample': 0.8}

Best CV MAE:
40.05499825602506

Performance on Test Set:
MAE: 34.88246651749027
R² : 0.7719643566245527


# Full size embeddings with sanitized SMILES (no wildcards)

In [26]:
X_clean = pd.read_csv("chemberta_clean_embeddings.csv", sep=",").set_index("_id")

print("Shape X:", X_clean.shape)
print("Shape y:", y.shape)
X_clean.sample(10)


Shape X: (304, 768)
Shape y: (304,)


,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,emb_9,...,emb_758,emb_759,emb_760,emb_761,emb_762,emb_763,emb_764,emb_765,emb_766,emb_767
_id,,,,,,,,,,,,,,,,,,,,,
698ddd4a8603d1724287a2f2,1.557992,1.385635,0.619190,-1.815705,0.657958,-0.122931,0.036420,-0.197040,-1.099252,0.342397,...,-1.100458,0.074247,0.074445,-0.700865,-0.693973,-0.458280,-0.897476,-0.947696,-0.253818,0.737196
698ddd4a8603d1724287a30e,0.594751,0.995822,0.781928,-0.840935,0.473620,0.301080,-0.283328,-0.760244,0.128598,0.228368,...,-0.370508,0.500652,-0.453413,-1.048006,-0.461502,0.073049,-0.483093,-0.290768,0.170451,0.582168
698ddd4a8603d1724287a394,0.648157,1.314737,0.234193,-0.155037,0.079479,0.449514,0.006168,-0.539072,-0.417547,-0.283133,...,-0.366576,0.421518,-0.490088,-0.813173,-0.805037,-0.441615,-0.746912,-0.226003,0.386734,0.387244
698ddd4a8603d1724287a354,0.443252,0.827227,0.751749,-1.232584,0.518586,0.088844,0.160108,-0.966889,0.312935,0.205559,...,-0.205446,0.187602,-1.212343,-0.499035,-0.831816,0.530234,-0.772581,-0.177795,-0.377822,0.280722
698ddd4a8603d1724287a2a8,1.790004,0.870483,0.647512,-0.797739,0.008586,-0.114332,-0.516906,-0.675644,-0.723830,0.476406,...,-0.741096,-0.013422,-0.578754,-1.264744,-0.437943,-0.254113,-0.776075,-0.963344,-0.081671,1.153702
698ddd4a8603d1724287a3af,0.364505,-0.244171,-0.200357,-0.110299,0.567120,-0.025535,-0.322445,-0.156021,-0.365385,-0.553441,...,0.333295,0.786320,0.104347,-0.593720,-0.510967,0.526293,-0.368002,0.216617,-0.272380,0.333533
698ddd4a8603d1724287a34f,0.460047,1.064529,0.458559,-1.780423,0.443313,0.560540,-0.194998,-0.225507,-0.449218,0.092549,...,-0.839242,0.336865,-0.990911,-0.730826,-0.870362,-0.435844,-0.477971,0.004557,-0.103434,0.287456
698ddd4a8603d1724287a2ac,1.150817,2.053180,-0.729387,-2.503035,0.995682,-0.616646,-0.291574,-0.195170,-0.030727,1.042071,...,-0.955008,1.325084,-0.594329,-0.177548,-0.486223,-0.861235,-0.844327,-0.141997,-0.685346,0.365196
698ddd4a8603d1724287a2f9,0.092961,-0.240247,-0.822971,-0.344081,0.028611,-0.763690,-0.351252,-0.068901,-0.582036,-0.552537,...,-0.655304,-0.342653,-1.081857,-1.186788,-0.033999,0.572983,-0.872413,-0.459384,0.376612,0.936989


In [27]:
from sklearn.model_selection import GridSearchCV, KFold

param_grid = {
    "n_estimators": [100, 200, 400],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "subsample": [0.8, 1.0]
}

gb = GradientBoostingRegressor(random_state=random_state)
X_train_clean, X_test_clean, y_train, y_test = train_test_split(X_clean, y, test_size=0.2,random_state=random_state)

# Cross-validation 5 fold
cv = KFold(n_splits=5, shuffle=True, random_state=random_state)

# Grid search
grid_search = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_clean, y_train)

# Risultati tuning
print("Best parameters found:")
print(grid_search.best_params_)

print("\nBest CV MAE:")
print(-grid_search.best_score_)

# Valutazione finale su test set
best_model = grid_search.best_estimator_
pred_clean = best_model.predict(X_test_clean)

mae = mean_absolute_error(y_test, pred_clean)
r2 = r2_score(y_test, pred_clean)

print("\nPerformance on Test Set:")
print(f"MAE: {mae}")
print(f"R² : {r2}")

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best parameters found:
{'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 400, 'subsample': 0.8}

Best CV MAE:
40.55672452536468

Performance on Test Set:
MAE: 36.67266583506289
R² : 0.7819421041974586


# Reduced embeddings with sanitized SMILES

In [28]:
X_clean_red = pd.read_csv("chemberta_clean_embeddings_reduced.csv", sep=",").set_index("_id")

print("Shape X:", X_clean_red.shape)
print("Shape y:", y.shape)
X_clean_red.sample(10)


Shape X: (304, 50)
Shape y: (304,)


,pc_0,pc_1,pc_2,pc_3,pc_4,pc_5,pc_6,pc_7,pc_8,pc_9,...,pc_40,pc_41,pc_42,pc_43,pc_44,pc_45,pc_46,pc_47,pc_48,pc_49
_id,,,,,,,,,,,,,,,,,,,,,
698ddd4a8603d1724287a2f4,2.155877,-11.693488,-2.755556,3.435108,-0.999754,-5.044492,4.763760,-0.723477,-1.257021,1.178692,...,-0.301727,-0.138168,0.162886,0.344736,-0.384905,0.297244,0.255447,-0.167139,0.066771,-0.512523
698ddd4a8603d1724287a349,4.440438,-4.904855,-4.232357,-2.263445,-1.336601,-0.530004,1.173616,1.377132,2.181124,-0.941970,...,-0.318409,0.038980,-0.519586,0.036436,-0.251365,-0.092288,-0.303578,0.354731,-0.320932,0.310073
698ddd4a8603d1724287a39b,5.381117,5.392105,3.924372,0.999473,0.019944,-1.942308,1.323806,1.711575,2.518970,-0.392132,...,0.579271,0.418999,-0.548823,-0.423604,0.355340,-0.653661,-0.477441,-0.315016,0.896141,-0.601217
698ddd4a8603d1724287a30f,3.884569,-1.809001,-6.152789,-4.653371,-1.051869,1.363729,0.239512,0.057101,1.763873,-1.168305,...,0.219099,0.027641,-0.126104,-0.121891,0.300799,0.066816,0.186844,0.296636,-0.223573,-0.316070
698ddd4a8603d1724287a365,-6.275240,5.088718,0.470916,1.674905,-2.014753,-3.356207,-0.601283,-3.063592,-0.521014,1.703684,...,-0.804926,0.340549,-0.211522,1.047454,0.110917,0.685972,-0.806872,0.588191,0.539605,-0.415953
698ddd4a8603d1724287a2e5,-2.582523,4.613218,0.226549,3.738465,-1.317616,-1.453123,-0.567426,2.155054,-0.566388,0.066588,...,-0.168085,0.398359,0.778029,0.159478,0.352697,-0.151830,-0.393719,0.458085,0.056264,0.191752
698ddd4a8603d1724287a2c0,-16.983036,-7.759956,8.035340,-5.973223,1.964145,0.714098,2.033817,0.950687,2.171337,-0.085416,...,-0.046210,-0.069906,-0.025638,-0.127865,0.027722,-0.020236,-0.220278,0.092204,0.059083,-0.082755
698ddd4a8603d1724287a3b1,4.934700,6.478235,5.652082,-1.664595,0.072084,0.208597,1.965507,-2.774546,-1.321439,0.316326,...,0.407985,0.377819,-0.897389,-0.225966,-0.366257,0.457769,0.400654,0.015596,-0.055893,-0.144229
698ddd4a8603d1724287a325,-7.864962,4.206471,-3.880920,9.457144,1.556717,5.501392,1.331205,-1.803082,0.519112,-2.486710,...,0.140899,0.510034,-0.441562,-0.232643,0.369052,-0.064185,0.382811,0.220068,-0.170215,-0.176345


In [29]:
from sklearn.model_selection import GridSearchCV, KFold

param_grid = {
    "n_estimators": [100, 200, 400],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "subsample": [0.8, 1.0]
}

gb = GradientBoostingRegressor(random_state=random_state)
X_train_clean_red, X_test_clean_red, y_train, y_test = train_test_split(X_clean_red, y, test_size=0.2,random_state=random_state)

# Cross-validation 5 fold
cv = KFold(n_splits=5, shuffle=True, random_state=random_state)

# Grid search
grid_search = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_clean_red, y_train)

# Risultati tuning
print("Best parameters found:")
print(grid_search.best_params_)

print("\nBest CV MAE:")
print(-grid_search.best_score_)

# Valutazione finale su test set
best_model = grid_search.best_estimator_
pred_clean_red = best_model.predict(X_test_clean_red)

mae = mean_absolute_error(y_test, pred_clean_red)
r2 = r2_score(y_test, pred_clean_red)

print("\nPerformance on Test Set:")
print(f"MAE: {mae}")
print(f"R² : {r2}")

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best parameters found:
{'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 200, 'subsample': 0.8}

Best CV MAE:
44.907852678883486

Performance on Test Set:
MAE: 41.92007514664607
R² : 0.6812691655681014


# With less aggressive PCA (200 columns)

## Unsanitized

In [30]:
X_less_red = pd.read_csv("chemberta_embeddings_less_reduced.csv", sep=",").set_index("_id")

print("Shape X:", X_less_red.shape)
print("Shape y:", y.shape)
X_less_red.sample(10)


Shape X: (304, 200)
Shape y: (304,)


,pc_0,pc_1,pc_2,pc_3,pc_4,pc_5,pc_6,pc_7,pc_8,pc_9,...,pc_190,pc_191,pc_192,pc_193,pc_194,pc_195,pc_196,pc_197,pc_198,pc_199
_id,,,,,,,,,,,,,,,,,,,,,
698ddd4a8603d1724287a2ec,-0.310178,4.953666,4.308843,2.294513,0.637318,-0.617478,1.572176,0.573516,-1.446845,0.915644,...,0.005320,0.010704,0.026886,0.028938,-0.006001,-0.034300,-0.011270,0.021484,-0.008577,0.006963
698ddd4a8603d1724287a370,-6.267945,0.996955,-1.253592,0.334339,2.563540,-0.500069,2.473517,0.124240,0.419900,0.304430,...,0.041394,0.055949,-0.047812,0.061485,-0.031301,0.022959,-0.007643,0.068218,-0.007246,0.033564
698ddd4a8603d1724287a2d5,2.951440,2.727730,2.376900,0.995883,-2.637034,-1.039636,1.023367,1.532612,1.300855,0.819944,...,0.005160,0.045405,-0.011799,0.024686,0.021977,0.049311,0.011145,-0.023467,0.017162,0.023760
698ddd4a8603d1724287a2fd,1.876162,2.552234,3.561752,1.159059,-2.788561,0.160205,0.737195,0.202728,0.742997,1.555259,...,0.070017,-0.017401,0.048636,0.026569,-0.009525,0.070397,-0.048719,0.000671,-0.035859,0.004084
698ddd4a8603d1724287a373,3.100868,-5.391602,-0.377893,0.997373,1.126668,0.443557,0.968918,0.983904,0.545764,0.223434,...,-0.082470,-0.016481,0.040514,-0.031037,-0.008637,0.033308,0.000807,-0.032018,-0.052814,0.046654
698ddd4a8603d1724287a391,-4.909596,-2.272715,4.033197,-5.027775,0.141082,2.773107,3.381766,-0.770471,0.646394,0.247853,...,0.040865,-0.030387,-0.012267,-0.039441,0.013104,-0.029270,-0.021478,0.020029,-0.001301,0.002717
698ddd4a8603d1724287a2b0,4.271859,2.030406,-0.912840,0.664272,-2.609919,-0.533838,2.233142,2.179017,0.566702,-1.098276,...,-0.052637,-0.045374,0.048990,-0.017418,-0.079484,0.022595,0.036118,0.006026,-0.017794,-0.019028
698ddd4a8603d1724287a36e,1.352260,-2.692035,4.962583,1.977567,2.036923,0.592642,-0.125979,0.046156,-2.822859,-1.248251,...,0.041460,-0.066611,0.059261,-0.049339,0.017222,0.013069,0.042344,0.012921,-0.013538,-0.015380
698ddd4a8603d1724287a30b,5.038214,-0.559803,0.064626,-0.822848,-0.331687,-0.118288,-3.422267,0.366646,1.933694,-2.588256,...,-0.003816,-0.018716,-0.006429,0.010110,-0.003984,-0.019574,-0.022010,-0.018931,0.050532,0.026715


In [31]:
from sklearn.model_selection import GridSearchCV, KFold

param_grid = {
    "n_estimators": [100, 200, 400],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "subsample": [0.8, 1.0]
}

gb = GradientBoostingRegressor(random_state=random_state)
X_train_less_red, X_test_less_red, y_train, y_test = train_test_split(X_less_red, y, test_size=0.2,random_state=random_state)

# Cross-validation 5 fold
cv = KFold(n_splits=5, shuffle=True, random_state=random_state)

# Grid search
grid_search = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_less_red, y_train)

# Risultati tuning
print("Best parameters found:")
print(grid_search.best_params_)

print("\nBest CV MAE:")
print(-grid_search.best_score_)

# Valutazione finale su test set
best_model = grid_search.best_estimator_
pred_less_red = best_model.predict(X_test_less_red)

mae = mean_absolute_error(y_test, pred_less_red)
r2 = r2_score(y_test, pred_less_red)

print("\nPerformance on Test Set:")
print(f"MAE: {mae}")
print(f"R² : {r2}")

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best parameters found:
{'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 400, 'subsample': 0.8}

Best CV MAE:
44.09689424643358

Performance on Test Set:
MAE: 37.676864707380076
R² : 0.7493423247137191


## Sanitized

In [32]:
X_clean_less_red = pd.read_csv("chemberta_clean_embeddings_less_reduced.csv", sep=",").set_index("_id")

print("Shape X:", X_clean_less_red.shape)
print("Shape y:", y.shape)
X_clean_less_red.sample(10)


Shape X: (304, 200)
Shape y: (304,)


,pc_0,pc_1,pc_2,pc_3,pc_4,pc_5,pc_6,pc_7,pc_8,pc_9,...,pc_190,pc_191,pc_192,pc_193,pc_194,pc_195,pc_196,pc_197,pc_198,pc_199
_id,,,,,,,,,,,,,,,,,,,,,
698ddd4a8603d1724287a2e9,4.337376,-1.803301,-5.985555,-1.909791,-1.037025,1.498258,1.319621,1.310074,2.948494,-0.237587,...,-0.107624,0.087027,-0.019043,-0.002249,0.038836,0.003507,-0.033082,-0.009326,-0.041923,-0.088649
698ddd4a8603d1724287a2d8,-5.004421,-2.772865,-5.161053,0.241783,1.367538,1.489346,-0.170222,4.777643,-3.711735,4.606418,...,-0.001937,0.027074,-0.009036,0.024522,-0.029524,-0.025270,0.003679,0.012759,0.014504,-0.011594
698ddd4a8603d1724287a370,4.931824,3.022549,4.399147,-1.054826,-2.060007,-2.817327,-2.429033,0.515002,2.134043,0.042425,...,-0.150889,-0.020452,-0.054702,-0.020024,0.044926,0.073138,0.044207,0.032635,-0.042236,-0.057952
698ddd4a8603d1724287a3a0,6.369398,5.064163,0.487642,-3.082035,-1.798559,1.555378,0.175531,0.956782,-0.026870,0.486430,...,0.146040,-0.089882,0.098305,-0.103302,-0.152203,0.015918,0.031413,0.092398,-0.016179,-0.191691
698ddd4a8603d1724287a31b,-11.843327,5.861573,1.134599,8.379112,1.035999,-0.190223,1.647332,-1.201885,0.500139,-3.439879,...,0.036186,0.062269,0.033735,-0.053265,-0.008382,-0.011363,-0.014316,0.005303,-0.000707,0.009666
698ddd4a8603d1724287a2ad,-2.828854,-4.312170,-5.762579,-3.645039,0.695471,0.776762,-1.911643,2.393643,-2.831511,-0.195093,...,-0.064796,0.043210,-0.027755,-0.011890,-0.015155,0.046652,0.110358,-0.010166,-0.040067,-0.007738
698ddd4a8603d1724287a2fe,0.968387,-0.889336,-1.525942,6.900988,4.011849,1.365075,0.757593,3.744479,0.208371,-2.286016,...,0.050338,-0.020614,-0.023425,0.008262,0.024901,0.016322,0.018944,0.001968,-0.009147,0.016561
698ddd4a8603d1724287a34f,-5.878883,5.428910,2.078943,2.709603,2.051320,-6.424239,-1.417353,-0.199201,-0.781882,-0.211293,...,0.063762,0.097917,-0.055028,0.136122,-0.093300,-0.032302,-0.047392,0.004510,-0.047903,0.025949
698ddd4a8603d1724287a3ae,8.720283,1.888135,4.091634,-3.814681,2.266021,2.028666,0.119381,-2.263109,-1.231627,-0.241758,...,-0.056994,-0.038685,-0.003139,-0.012715,-0.016949,-0.044343,0.025132,-0.057923,0.026686,-0.017056


In [33]:
from sklearn.model_selection import GridSearchCV, KFold

param_grid = {
    "n_estimators": [100, 200, 400],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "subsample": [0.8, 1.0]
}

gb = GradientBoostingRegressor(random_state=random_state)
X_train_clean_less_red, X_test_clean_less_red, y_train, y_test = train_test_split(X_clean_less_red, y, test_size=0.2,random_state=random_state)

# Cross-validation 5 fold
cv = KFold(n_splits=5, shuffle=True, random_state=random_state)

# Grid search
grid_search = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_clean_less_red, y_train)

# Risultati tuning
print("Best parameters found:")
print(grid_search.best_params_)

print("\nBest CV MAE:")
print(-grid_search.best_score_)

# Valutazione finale su test set
best_model = grid_search.best_estimator_
pred_clean_less_red = best_model.predict(X_test_clean_less_red)

mae = mean_absolute_error(y_test, pred_clean_less_red)
r2 = r2_score(y_test, pred_clean_less_red)

print("\nPerformance on Test Set:")
print(f"MAE: {mae}")
print(f"R² : {r2}")

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best parameters found:
{'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 400, 'subsample': 0.8}

Best CV MAE:
48.50321249698794

Performance on Test Set:
MAE: 44.067125335091056
R² : 0.6013702199260149
